# 02 — Transformação dos Dados | Camada Silver

Neste notebook são realizadas as transformações necessárias para padronizar os dados carregados anteriormente na camada Bronze.
Os quatro conjuntos possuem a mesma estrutura geral, mas representam indicadores e sexos diferentes. Nesta etapa eles serão organizados em um único formato, com tipos de dados adequados e campos que identifiquem o sexo e o indicador analisado.

## 1. Leitura das tabelas Bronze

As quatro tabelas criadas na etapa anterior são carregadas a partir do catálogo do Databricks para iniciar o processo de transformação.

In [0]:
from pyspark.sql import functions as F

df_obesity_female = spark.table(
    "workspace.default.bronze_obesity_female"
)

df_overweight_female = spark.table(
    "workspace.default.bronze_overweight_female"
)

df_obesity_male = spark.table(
    "workspace.default.bronze_obesity_male"
)

df_overweight_male = spark.table(
    "workspace.default.bronze_overweight_male"
)

## 2. Padronização da estrutura

Na camada Bronze os valores foram mantidos como texto e cada arquivo foi armazenado separadamente.
Para a camada Silver, os nomes das colunas são padronizados e os campos `year` e `prevalence_pct` são convertidos para tipos numéricos.
Também são adicionadas as colunas `sex` e `indicator`, permitindo identificar a origem de cada registro após a união dos conjuntos.

In [0]:
def padronizar_dataset(df, sex, indicator):
    return (
        df.select(
            F.col("entity").alias("country"),
            F.col("code").alias("country_code"),
            F.col("year").cast("int").alias("year"),
            F.lit(sex).alias("sex"),
            F.lit(indicator).alias("indicator"),
            F.col("raw_value").cast("double").alias("prevalence_pct")
        )
    )

In [0]:
silver_obesity_female = padronizar_dataset(
    df_obesity_female,
    "Female",
    "Obesity"
)

silver_overweight_female = padronizar_dataset(
    df_overweight_female,
    "Female",
    "Overweight"
)

silver_obesity_male = padronizar_dataset(
    df_obesity_male,
    "Male",
    "Obesity"
)

silver_overweight_male = padronizar_dataset(
    df_overweight_male,
    "Male",
    "Overweight"
)

## 3. União dos conjuntos de dados

Após a padronização, os quatro conjuntos passam a possuir exatamente o mesmo esquema.
Com isso, os registros podem ser reunidos em uma única tabela Silver, facilitando as análises posteriores.

In [0]:
silver_nutrition = (
    silver_obesity_female
    .unionByName(silver_overweight_female)
    .unionByName(silver_obesity_male)
    .unionByName(silver_overweight_male)
)

In [0]:
display(silver_nutrition.limit(20))

## 4. Verificação do esquema

Após as transformações, é feita uma nova inspeção do esquema para confirmar se os campos numéricos foram convertidos corretamente.

In [0]:
silver_nutrition.printSchema()

## 5. Análise inicial de qualidade dos dados

Antes de persistir a camada Silver, são realizadas algumas verificações básicas de qualidade.
Nesta etapa são analisados valores ausentes, duplicidades, intervalo temporal e possíveis valores de prevalência fora do intervalo esperado de 0% a 100%.

In [0]:
print("Quantidade total de registros:", silver_nutrition.count())

In [0]:
display(
    silver_nutrition.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in silver_nutrition.columns
    ])
)

### Verificação de registros duplicados

Também é verificado se existem registros completamente duplicados após a unificação das quatro fontes.

In [0]:
duplicados = (
    silver_nutrition
    .groupBy(
        "country",
        "country_code",
        "year",
        "sex",
        "indicator",
        "prevalence_pct"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Quantidade de registros duplicados:", duplicados.count())

In [0]:
silver_nutrition.select(
    F.min("year").alias("ano_minimo"),
    F.max("year").alias("ano_maximo")
).show()

In [0]:
valores_invalidos = silver_nutrition.filter(
    (F.col("prevalence_pct") < 0) |
    (F.col("prevalence_pct") > 100)
)

print(
    "Registros com prevalência fora de 0 a 100:",
    valores_invalidos.count()
)

In [0]:
display(
    silver_nutrition
    .groupBy("indicator", "sex")
    .count()
    .orderBy("indicator", "sex")
)

### Resultado da análise de qualidade
As verificações realizadas não identificaram problemas relevantes de qualidade no conjunto de dados.
Não foram encontrados valores nulos ou registros duplicados, e todos os valores de prevalência estão dentro do intervalo esperado de 0% a 100%. O período disponível nos dados vai de 1980 a 2024.
Também foi verificado que os quatro grupos analisados, obesidade feminina, obesidade masculina, excesso de peso feminino e excesso de peso masculino, possuem a mesma quantidade de registros.
Por se tratar de uma fonte de dados já estruturada e disponibilizada para análise, não foi necessário realizar exclusões ou imputações de valores nesta etapa. As principais transformações realizadas na camada Silver foram a padronização do esquema, conversão dos tipos de dados e consolidação das quatro fontes.

## 6. Persistência da camada Silver

Após a padronização e as verificações de qualidade, o conjunto consolidado é armazenado como uma tabela Delta.
A partir desta tabela será construída a camada Gold, voltada especificamente para responder às perguntas definidas no projeto.

In [0]:
silver_nutrition.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_nutrition")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.default.silver_nutrition
        LIMIT 20
    """)
)

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.default
    """)
)

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.default
    """)
)

## Conclusão

Nesta etapa os quatro conjuntos de dados da camada Bronze foram padronizados e reunidos em uma única estrutura.
Os tipos dos campos foram ajustados, as informações de sexo e indicador foram explicitadas e foram realizadas verificações básicas de qualidade, incluindo valores ausentes, duplicidades e valores fora dos intervalos esperados.
O resultado foi persistido na tabela `silver_nutrition`, que será utilizada como base para a construção da camada Gold e para as análises do projeto.